# Momentum 12–1: Top N e Deadband

Cópia revisada de `teste robusto(2).ipynb`. Execute de cima para baixo.

- Histórico inicial: **2015–2016**, somente para formar sinais.
- Treinamento: **2017–2020**. Validação: **2021**, sem ajuste de parâmetros com base nela.
- Estratégias: Top 5/10/15 e Deadband 7/10, ambos com cinco posições.
- Não há GARCH. Dados posteriores a 2021 são excluídos antes de calcular retornos ou sinais.

Convenções preservadas: Momentum = preço defasado 21 pregões / preço defasado
252 pregões − 1; sinal no último pregão disponível do mês; ordem executada
**no fechamento do pregão seguinte**, com exposição nova a partir do pregão posterior.
O backtest começa em caixa em 2017, com primeiro sinal no fim de janeiro.
As posições e a memória do Deadband continuam na passagem de 2020 para 2021.
Os pesos variam com os retornos entre rebalanceamentos e voltam a ser iguais a cada ordem.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
import ast
import json

ARQUIVO_PRECOS = Path(r"C:\Users\julio\PBL FGV\FGV QUANT\treinee\TraineeFGVquant\bases de dados\economatica-3.xlsx")
INICIO_HISTORICO = '2015-01-01'
INICIO_BACKTEST = '2017-01-01'
FIM_DADOS = '2021-12-31'
PERIODOS = {'Treinamento (2017–2020)': ('2017-01-01', '2020-12-31'),
            'Validação (2021)': ('2021-01-01', '2021-12-31')}
CUSTOS = (0.0, 0.001, 0.003)
CUSTO_BASE = 0.001


## Funções de preparação e seleção

Mantido o preenchimento de no máximo uma observação ausente, sem usar dados futuros.
Datas/tickers duplicados e preços não positivos geram erro para evitar correções silenciosas.
Meses com menos ativos elegíveis que o necessário não geram ordem: a carteira anterior
é mantida (ou permanece em caixa). Os empates seguem a ordem original das colunas.


In [ ]:
def preparar_precos(base, inicio, fim):
    precos = base.copy()
    precos.columns = ['Data'] + [str(c).strip().split('\n')[-1].strip()
                                for c in precos.columns[1:]]
    if precos.columns.duplicated().any():
        raise ValueError('Tickers duplicados após normalização do cabeçalho.')
    precos['Data'] = pd.to_datetime(precos['Data'], errors='coerce')
    precos = precos.dropna(subset=['Data']).set_index('Data').sort_index()
    precos = precos.loc[inicio:fim]
    if precos.index.has_duplicates:
        raise ValueError('Datas duplicadas na planilha.')
    precos = precos.replace('-', np.nan).apply(pd.to_numeric, errors='coerce')
    precos = precos.dropna(how='all').ffill(limit=1)
    if precos.empty or (precos <= 0).any().any() or np.isinf(precos.to_numpy()).any():
        raise ValueError('Base vazia ou com preços não positivos/infinitos.')
    return precos


def criar_top_n(momentum_mensal, n):
    if not isinstance(n, int) or n < 1:
        raise ValueError('n deve ser inteiro positivo.')
    historico = {}
    for data, linha in momentum_mensal.iterrows():
        ranking = linha.dropna().sort_values(ascending=False, kind='stable')
        if len(ranking) >= n:
            historico[data] = ranking.head(n).index.tolist()
    return pd.DataFrame.from_dict(historico, orient='index',
                                  columns=[f'Ativo {i}' for i in range(1, n + 1)])


def criar_deadband(momentum_mensal, limite, n=5):
    if not isinstance(n, int) or not isinstance(limite, int) or n < 1 or limite < n:
        raise ValueError('Use limite inteiro >= n >= 1.')
    carteira, historico = [], {}
    for data, linha in momentum_mensal.iterrows():
        ranking = linha.dropna().sort_values(ascending=False, kind='stable')
        if len(ranking) < n:
            continue
        permitidos = ranking.head(limite).index
        carteira = [ativo for ativo in carteira if ativo in permitidos]
        for ativo in ranking.index:
            if len(carteira) == n:
                break
            if ativo not in carteira:
                carteira.append(ativo)
        historico[data] = carteira.copy()
    return pd.DataFrame.from_dict(historico, orient='index',
                                  columns=[f'Ativo {i}' for i in range(1, n + 1)])


def criar_pesos(carteira, universo):
    pesos = pd.DataFrame(0.0, index=carteira.index, columns=universo)
    for data, linha in carteira.iterrows():
        ativos = linha.dropna().tolist()
        if not ativos or len(set(ativos)) != len(ativos):
            raise ValueError(f'Carteira vazia ou com duplicatas em {data}.')
        if not set(ativos).issubset(pesos.columns):
            raise ValueError(f'Ativo fora do universo em {data}.')
        pesos.loc[data, ativos] = 1.0 / len(ativos)
    return pesos


## Backtest, custos e métricas

Turnover é a **soma absoluta das compras e vendas**, sem dividir por dois;
a entrada inicial também tem custo. Mantida a fórmula líquida multiplicativa
`(1 + retorno bruto) × (1 − custo × turnover) − 1`.
O custo é uma taxa por valor negociado (0%, 0,10% ou 0,30%). Não se cobra liquidação
artificial ao fim de cada período. Sharpe assume taxa livre de risco zero e 252 pregões/ano.
Retornos ausentes de posições compradas interrompem a execução com data e ativos;
não são tratados como retorno zero. O drawdown inclui o patrimônio inicial de 1.


In [ ]:
def backtest(pesos_alvo, retornos, inicio, fim):
    r_periodo = retornos.loc[inicio:fim]
    if r_periodo.empty or not retornos.index.is_monotonic_increasing or retornos.index.has_duplicates:
        raise ValueError('Retornos vazios ou datas fora de ordem/duplicadas.')
    if not pesos_alvo.columns.equals(retornos.columns):
        raise ValueError('Colunas dos pesos e retornos devem coincidir.')
    if pesos_alvo.index.has_duplicates or not pesos_alvo.index.isin(r_periodo.index).all():
        raise ValueError('Sinais duplicados ou fora dos pregões do backtest.')
    if (not np.isfinite(pesos_alvo.to_numpy()).all() or (pesos_alvo < 0).any().any()
            or not np.allclose(pesos_alvo.sum(axis=1), 1)):
        raise ValueError('Pesos devem ser finitos, não negativos e somar um.')
    ordens = pesos_alvo.reindex(r_periodo.index).shift(1)
    pesos_atuais = pd.Series(0.0, index=retornos.columns)
    historico = []
    for data, r in r_periodo.iterrows():
        faltantes = r.index[(pesos_atuais > 0) & r.isna()].tolist()
        if faltantes:
            raise ValueError(f'Retorno ausente em {data.date()}: {faltantes}')
        r = r.fillna(0.0)
        if not np.isfinite(r.to_numpy()).all() or (r < -1).any():
            raise ValueError(f'Retorno inválido em {data}.')
        retorno_dia = float((pesos_atuais * r).sum())
        if retorno_dia <= -1:
            raise ValueError(f'Patrimônio esgotado em {data}.')
        pesos_atuais = pesos_atuais * (1 + r) / (1 + retorno_dia)
        turnover = 0.0
        if ordens.loc[data].notna().all():
            novos_pesos = ordens.loc[data]
            turnover = float((novos_pesos - pesos_atuais).abs().sum())
            pesos_atuais = novos_pesos.copy()
        historico.append({'Data': data, 'Retorno': retorno_dia, 'Turnover': turnover})
    return pd.DataFrame(historico).set_index('Data')


def aplicar_custos(resultado, custo):
    if not np.isfinite(custo) or custo < 0 or (custo * resultado['Turnover'] >= 1).any():
        raise ValueError('Custo inválido ou superior ao patrimônio.')
    return (1 + resultado['Retorno']) * (1 - custo * resultado['Turnover']) - 1


def calcular_sharpe(retornos, pregoes_ano=252):
    volatilidade = retornos.std(ddof=1)
    if not np.isfinite(volatilidade) or volatilidade <= 1e-15:
        return np.nan
    return retornos.mean() / volatilidade * np.sqrt(pregoes_ano)


def calcular_metricas(resultado, custo=0.001, pregoes_ano=252):
    if resultado.empty or resultado[['Retorno', 'Turnover']].isna().any().any():
        raise ValueError('Resultado vazio ou com valores ausentes.')
    r = aplicar_custos(resultado, custo)
    patrimonio = (1 + r).cumprod()
    retorno_anual = patrimonio.iloc[-1] ** (pregoes_ano / len(r)) - 1
    pico = patrimonio.cummax().clip(lower=1.0)
    return {'Retorno anual (%)': retorno_anual * 100,
            'Volatilidade (%)': r.std(ddof=1) * np.sqrt(pregoes_ano) * 100,
            'Sharpe': calcular_sharpe(r, pregoes_ano),
            'Max Drawdown (%)': (patrimonio / pico - 1).min() * 100,
            'Turnover anual': resultado['Turnover'].sum() * pregoes_ano / len(r)}


## Leitura dos preços e construção do Momentum 12–1


In [ ]:
if not ARQUIVO_PRECOS.is_file():
    raise FileNotFoundError(f'Planilha de preços não encontrada: {ARQUIVO_PRECOS}. '
                            'Ajuste ARQUIVO_PRECOS antes de executar; dados reais não são substituídos por simulação.')
base = pd.read_excel(ARQUIVO_PRECOS, header=3)
precos = preparar_precos(base, INICIO_HISTORICO, FIM_DADOS)
retornos = precos.pct_change(fill_method=None)
momentum = precos.shift(21) / precos.shift(252) - 1
momentum_mensal = momentum.groupby(momentum.index.to_period('M')).tail(1)
momentum_mensal = momentum_mensal.loc[INICIO_BACKTEST:FIM_DADOS]
print(f'Preços utilizados: {precos.index.min().date()} a {precos.index.max().date()}; '
      f'{len(precos)} observações; {precos.shape[1]} ativos.')
display(momentum_mensal.count(axis=1).rename('Ativos elegíveis por mês').to_frame())


## Seleções, pesos e execução das cinco estratégias


In [ ]:
carteiras = {f'Top {n}': criar_top_n(momentum_mensal, n) for n in (5, 10, 15)}
carteiras.update({f'Deadband {limite}': criar_deadband(momentum_mensal, limite)
                  for limite in (7, 10)})
pesos = {nome: criar_pesos(carteira, retornos.columns) for nome, carteira in carteiras.items()}
resultados = {nome: backtest(alvos, retornos, INICIO_BACKTEST, FIM_DADOS)
              for nome, alvos in pesos.items()}
display(pd.DataFrame({'Meses com sinal': {nome: len(c) for nome, c in carteiras.items()},
                      'Meses sem sinal': {nome: len(momentum_mensal) - len(c)
                                           for nome, c in carteiras.items()}}))


## Métricas separadas por período e sensibilidade a custos


In [ ]:
amostras = {periodo: {nome: resultado.loc[inicio:fim].copy()
                      for nome, resultado in resultados.items()}
            for periodo, (inicio, fim) in PERIODOS.items()}
tabelas_metricas = {}
linhas_custos = []
for periodo, estrategias in amostras.items():
    tabela = pd.DataFrame({nome: calcular_metricas(resultado, CUSTO_BASE)
                           for nome, resultado in estrategias.items()}).T
    tabelas_metricas[periodo] = tabela
    print(f'{periodo} — custo de {CUSTO_BASE:.2%} por valor negociado')
    display(tabela.round(3))
    for nome, resultado in estrategias.items():
        for custo in CUSTOS:
            linhas_custos.append({'Período': periodo, 'Estratégia': nome,
                                  'Custo': custo, **calcular_metricas(resultado, custo)})
sensibilidade_custos = pd.DataFrame(linhas_custos).set_index(['Período', 'Estratégia', 'Custo'])
display(sensibilidade_custos.round(3))


## Patrimônio líquido por período (base 1)


In [ ]:
for periodo, estrategias in amostras.items():
    patrimonio = pd.DataFrame({nome: (1 + aplicar_custos(resultado, CUSTO_BASE)).cumprod()
                               for nome, resultado in estrategias.items()})
    ax = patrimonio.plot(figsize=(12, 5), title=f'{periodo} — custo {CUSTO_BASE:.2%}')
    ax.set(xlabel='Data', ylabel='Patrimônio líquido (base 1)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Limitações e interpretação

- A planilha é uma dependência externa, não incorporada ao notebook. Em outro computador,
  ajuste `ARQUIVO_PRECOS`. Sem ela, apenas as definições de funções podem executar;
  não é possível reproduzir ou confirmar resultados financeiros reais.
- O arquivo pode conter outros anos, mas só o intervalo 2015–2021 entra nos cálculos.
  A frequência é a das linhas da planilha, assumidas como pregões, sem validação por calendário de bolsa.
- O preenchimento de um preço ausente foi preservado; pode mascarar um dia sem negociação.
  Lacunas maiores em posições compradas interrompem o backtest. Não há tratamento de delisting.
- Não foi auditado se os preços são ajustados por dividendos/desdobramentos, se o universo
  tem viés de sobrevivência ou se os dados estavam disponíveis historicamente.
- Não há filtros de liquidez, restrições de lote, impacto de mercado ou impostos.
  Custos são proporcionais ao turnover; os pesos posteriores ao custo são idealizados como os pesos-alvo.
- A validação mantém as posições anteriores e não faz nova entrada artificial em janeiro de 2021.
  Cada curva é renormalizada para seu próprio período. Não há escolha automática da melhor estratégia.
- As métricas anualizam pela quantidade de observações (252/ano), não pelo tempo de calendário.
  Sharpe de série constante é indefinido (`NaN`). Resultados não constituem previsão de desempenho.


## Verificação desta cópia

Em 27/10/2026, todas as 7 células de código foram executadas em sequência em um processo Python novo, com namespace compartilhado e captura de tabelas e gráficos, usando a planilha local original. Não foi usado um kernel Jupyter para essa verificação. Também passaram testes sintéticos de regressão para Top N, Deadband sem mudanças e com substituição, atraso de execução, turnover inicial, custos, ausência de retorno, anualização e drawdown. O hash SHA-256 do notebook original permaneceu inalterado.


In [ ]:
janelas = {
    "6-1": 126,
    "9-1": 189,
    "12-1": 252
}

momentums = {}

for nome, dias in janelas.items():

    indicador = precos.shift(21) / precos.shift(dias) - 1

    momentum_mensal_novo = indicador.groupby(
        indicador.index.to_period("M")
    ).tail(1)

    momentum_mensal_novo = momentum_mensal_novo.loc[
        INICIO_BACKTEST:FIM_DADOS
    ]

    momentums[nome] = momentum_mensal_novo

In [ ]:
carteiras_grid = {}

for janela, indicador in momentums.items():

    # Estratégias Top N
    for n in [5, 10, 15]:

        nome = f"{janela} | Top {n}"

        carteiras_grid[nome] = criar_top_n(
            indicador, n
        )

    # Estratégias Deadband
    for limite in [7, 10]:

        nome = f"{janela} | Deadband {limite}"

        carteiras_grid[nome] = criar_deadband(
            indicador, limite
        )

print("Quantidade de estratégias:", len(carteiras_grid))

In [ ]:
pesos_grid = {}

for nome, carteira in carteiras_grid.items():

    pesos_grid[nome] = criar_pesos(
        carteira,
        retornos.columns
    )

In [ ]:
resultados_grid = {}

for nome, pesos_alvo in pesos_grid.items():

    resultados_grid[nome] = backtest(
        pesos_alvo,
        retornos,
        INICIO_BACKTEST,
        FIM_DADOS
    )

print("Backtests executados:", len(resultados_grid))

In [ ]:
linhas = []

for nome, resultado in resultados_grid.items():

    # Selecionar apenas o treinamento
    treino = resultado.loc["2017":"2020"]

    # Calcular métricas com custo de 0,10%
    metricas = calcular_metricas(
        treino,
        custo=CUSTO_BASE
    )

    janela, selecao = nome.split(" | ")

    linhas.append({
        "Janela": janela,
        "Seleção": selecao,
        **metricas
    })

tabela_grid_treino = pd.DataFrame(linhas)

# Ordenar pelo Sharpe
tabela_grid_treino = tabela_grid_treino.sort_values(
    "Sharpe",
    ascending=False
)

display(tabela_grid_treino.round(3))

In [ ]:
matriz_sharpe = tabela_grid_treino.pivot(
    index="Janela",
    columns="Seleção",
    values="Sharpe"
)

matriz_sharpe = matriz_sharpe.reindex(
    index=["6-1", "9-1", "12-1"],
    columns=[
        "Top 5",
        "Top 10",
        "Top 15",
        "Deadband 7",
        "Deadband 10"
    ]
)

display(matriz_sharpe.round(3))

In [ ]:
custos_teste = {
    "0%": 0,
    "0,10%": 0.001,
    "0,30%": 0.003
}

linhas = []

for nome, resultado in resultados_grid.items():

    treino = resultado.loc["2017":"2020"]

    linha = {"Estratégia": nome}

    for custo_nome, custo_valor in custos_teste.items():

        metricas = calcular_metricas(
            treino,
            custo=custo_valor
        )

        linha[f"Sharpe {custo_nome}"] = metricas["Sharpe"]

    linhas.append(linha)

tabela_robustez = pd.DataFrame(linhas).set_index("Estratégia")

display(tabela_robustez.round(3))

esse aqui é pra importar as funçoes q eu usei pra fazer o momentum, ai usando elas pro garch agnt evita divergencias metodologicas

In [ ]:
raiz = Path.cwd().resolve()
while not (raiz/ ".git").exists():
    if raiz == raiz.parent:
        raise FileNotFoundError("nao existe")
    raiz = raiz.parent

arquivo = raiz / "notebooks" / "momentum apenas" / "momentums.ipynb"
notebook = json.loads(arquivo.read_text(encoding="utf-8"))

nomes =  [
    "preparar_precos",
    "criar_top_n",
    "criar_deadband",
    "criar_pesos",
    "backtest",
    "aplicar_custos",
    "calcular_sharpe",
    "calcular_metricas"
]

funcoes = {}

for celula in notebook["cells"]:
    if celula ["cell_type"] != "code":
        continue
    codigo = "".join(celula["source"])
    arvore = ast.parse(codigo)

    for no in arvore.body:
       if isinstance(no, ast.FunctionDef) and no.name in nomes:
         funcoes[no.name] = ast.get_source_segment(codigo, no)
faltantes = set(nomes) - set(funcoes)

if faltantes:
 raise ValueError(f"faltam:{faltantes}")

conteudo = (
    "import numpy as np\n"
    "import pandas as pd\n\n\n"
    + "\n\n\n".join(funcoes[nome] for nome in nomes)
    + "\n")

destino = raiz / "funcoes_momentum.py"
destino.write_text(conteudo, encoding="utf8")


print("foi")
print(len(funcoes))

   

    
